# Neural mechanisms, optimization and model choice
**Objective:** calculate a forward pass and gradient, inspect dimensions, train with explicit validation selection, and compare against classical alternatives.
Neural networks are candidates, not a mandatory endpoint. The numerical network below is illustrative.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


In [ ]:
import torch
from torch import nn
z=torch.tensor([[1.,-1.]])
W1=torch.tensor([[1.,1.],[1.,-1.]])
w=torch.tensor([1.,2.],requires_grad=True)
hidden=torch.relu(z@W1.T)
prediction=100+hidden@w
loss=(prediction-106).square().mean();loss.backward()
print('Hidden:',hidden,'prediction:',prediction.detach(),'loss:',loss.detach(),'gradient:',w.grad)
np.testing.assert_allclose(w.grad.numpy(),[0.,-8.])


## 1. Prepare a real fitting/validation/test population
Medellín supplies contemporaneous environmental estimation, not future weather forecasting. We use the immutable hourly snapshot and display every feature role. All scales are learned on training data.

In [ ]:
from sklearn.preprocessing import StandardScaler
case=MATCHED/'medellin_hourly_environment'
a=np.load(case/'examples.npz',allow_pickle=False);X,y=a['X'],a['y']
tr,va,te=[np.flatnonzero(a['split']==s) for s in ['train','validation','test']]
meta=json.loads((case/'case.json').read_text());display(pd.Series(meta['features'],name='features'))
sx,sy=StandardScaler().fit(X[tr]),StandardScaler().fit(y[tr])
x=torch.tensor(sx.transform(X[tr]),dtype=torch.float32);target=torch.tensor(sy.transform(y[tr]),dtype=torch.float32)
xv=torch.tensor(sx.transform(X[va]),dtype=torch.float32)
assert not {'rssi','snr','experimental_pl'} & set(meta['features'])


## 2. See the complete training loop
This compact classroom fit uses up to 40 epochs and patience 8, deliberately distinct from the archived 80/12 experiment. It demonstrates selection, not exact reproduction of that configuration. We choose by validation MAE in dB; training minimizes standardized MSE.

In [ ]:
import copy
torch.set_num_threads(2);torch.manual_seed(42)
net=nn.Sequential(nn.Linear(X.shape[1],64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
optimizer=torch.optim.Adam(net.parameters(),lr=.001,weight_decay=.0001)
loss_fn=nn.MSELoss();best=np.inf;best_epoch=0;history=[]
for epoch in range(1,41):
    net.train();training_loss=0.
    for idx in torch.randperm(len(x)).split(256):
        optimizer.zero_grad()
        loss=loss_fn(net(x[idx]),target[idx]);loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),5.)
        optimizer.step();training_loss+=float(loss.detach())*len(idx)
    net.eval()
    with torch.no_grad():pred=sy.inverse_transform(net(xv).numpy())
    validation_mae=np.abs(y[va]-pred).mean()
    history.append((epoch,training_loss/len(x),validation_mae))
    if validation_mae<best-1e-6:
        best=validation_mae;best_epoch=epoch;best_state=copy.deepcopy(net.state_dict())
    if epoch-best_epoch>=8:break
net.load_state_dict(best_state)
print('Selected epoch:',best_epoch,'Parameters:',sum(v.numel() for v in net.parameters()))


In [ ]:
curve=pd.DataFrame(history,columns=['epoch','train_mse_scaled','validation_mae_db'])
fig,axs=plt.subplots(1,2,figsize=(8,3))
axs[0].plot(curve.epoch,curve.train_mse_scaled);axs[0].set(xlabel='Epoch',ylabel='Training MSE (standardized)')
axs[1].plot(curve.epoch,curve.validation_mae_db);axs[1].axvline(best_epoch,linestyle='--',color='black');axs[1].set(xlabel='Epoch',ylabel='Validation MAE (dB)');plt.show()
net.eval()
with torch.no_grad():test_prediction=sy.inverse_transform(net(torch.tensor(sx.transform(X[te]),dtype=torch.float32)).numpy())
print('Classroom test MAE dB:',np.abs(y[te]-test_prediction).mean())


## 3. Compare actual alternatives and inspect a temporal filter

In [ ]:
m=pd.read_csv(GUIDE/'metrics.csv')
display(m[m.case.str.startswith('environment_')].groupby(['case','model']).rmse.agg(['mean','std']))
sequence=np.array([1.,2.,3.,4.,5.]);kernel=np.array([.25,.5,.25])
filtered=np.array([sequence[i:i+3]@kernel for i in range(3)])
np.testing.assert_allclose(filtered,[2.,3.,4.]);print(filtered)


**Exercise:** explain why a stack of affine maps without activations remains affine. What does three initialization seeds fail to measure?

<details><summary>Feedback after your attempt</summary>

multiplying and adding affine maps yields another affine map. Seeds vary optimization choices, not city, device or data collection. A model that wins under one initialization should not be selected from test scores.

</details>

**Adaptation task:** compare the MLP with Ridge, RF and boosting under a fixed validation budget. Change architecture only after diagnosing representation, scale and learning curves.